# Bài 9: Biến dạng ảnh, Ánh xạ ngược và Nội suy song tuyến tính (Image Warping, Inverse Mapping, and Bilinear Interpolation)

Bài 8 đã xây dựng các ma trận biến đổi; bài này nói về việc thực sự *áp dụng* một ma trận để lấy mẫu lại (resample) ảnh. Cách ngây thơ để làm điều này &mdash; đẩy mỗi điểm ảnh nguồn tới vị trí sau biến đổi của nó &mdash; hóa ra là sai. Chúng ta sẽ thấy tại sao, khắc phục bằng **ánh xạ ngược (inverse mapping)**, rồi xử lý việc ánh xạ ngược rơi vào các tọa độ điểm ảnh lẻ (phân số) cần phải được *nội suy (interpolate)*.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

In [ ]:
def make_test_image(size=60):
    img = np.zeros((size, size, 3), dtype=np.uint8)
    cv2.rectangle(img, (5, 5), (size - 5, size - 5), (60, 90, 160), -1)   # muted blue
    tri = np.array([[30, 12], [10, 32], [50, 32]], dtype=np.int32)
    cv2.fillPoly(img, [tri], (225, 195, 60))  # muted yellow
    return img

img = make_test_image()
plt.imshow(img)
plt.title('Test image')
plt.axis('off')
plt.show()

## Vấn đề của ánh xạ xuôi (The problem with forward mapping)
Cách hiển nhiên để biến dạng một ảnh: với mỗi điểm ảnh nguồn $(x,y)$, tính điểm đích $(x',y')=M(x,y)$ và sao chép giá trị điểm ảnh tới đó. Cách này gọi là **ánh xạ xuôi (forward mapping)**.

Nó có một khiếm khuyết cơ bản: trong khi các điểm ảnh nguồn tạo thành một lưới *đầy đủ*, các vị trí sau biến đổi của chúng nói chung thì không. Khi phép biến đổi phóng to ảnh (hoặc xoay ảnh, vốn kéo giãn lưới cục bộ theo đường chéo), các vị trí đích tản ra và để lại khoảng trống &mdash; không có điểm ảnh nguồn nào tình cờ rơi đúng vào một số điểm ảnh đích.

In [ ]:
h, w = img.shape[:2]
M = cv2.getRotationMatrix2D((w / 2, h / 2), angle=25, scale=1.6)

forward = np.zeros_like(img)
ys, xs = np.mgrid[0:h, 0:w]
src_pts = np.stack([xs.ravel(), ys.ravel(), np.ones(xs.size)])
dst_pts = M @ src_pts

dxi = np.round(dst_pts[0]).astype(int)
dyi = np.round(dst_pts[1]).astype(int)
valid = (dxi >= 0) & (dxi < w) & (dyi >= 0) & (dyi < h)
forward[dyi[valid], dxi[valid]] = img[ys.ravel()[valid], xs.ravel()[valid]]

filled = np.count_nonzero(forward.sum(axis=2))
print(f'{filled} / {h*w} destination pixels received a value ({100*filled/(h*w):.0f}%)')

plt.imshow(forward)
plt.title('Forward mapping: visible holes (black speckles)')
plt.axis('off')
plt.show()

Ảnh động này minh họa ánh xạ xuôi: quét ảnh nguồn theo từng hàng và "vẩy" (splat) mỗi điểm ảnh vào ảnh đích. Hãy quan sát các lỗ hổng (các đốm đen) xuất hiện khi phép biến đổi làm các vị trí đích tản ra.

<img src="../animations/forward_mapping_demo.gif" alt="Ảnh động biến dạng bằng ánh xạ xuôi, quét ảnh nguồn theo từng hàng và vẩy điểm ảnh vào ảnh đích, cho thấy các lỗ hổng xuất hiện khi vị trí đích tản ra" style="display: block; margin: 0 auto; max-width: 680px;">

## Cách khắc phục: ánh xạ ngược (The fix: inverse mapping)
Thay vì hỏi "mỗi điểm ảnh nguồn đi về đâu?", hãy đặt câu hỏi ngược lại cho mỗi điểm ảnh *đích*: "điểm này đến từ đâu trong ảnh nguồn?" Điều đó có nghĩa là áp dụng phép biến đổi **ngược** $M^{-1}$ cho mỗi tọa độ đích. Vì giờ ta duyệt qua một lưới đích đầy đủ, mọi điểm ảnh đầu ra đều chắc chắn nhận được giá trị &mdash; không có lỗ hổng, theo cách xây dựng.

Vướng mắc là: $M^{-1}(x',y')$ hầu như không bao giờ rơi đúng vào một tọa độ nguồn nguyên. Ta cần *nội suy* một giá trị từ các điểm ảnh nguồn xung quanh.

In [ ]:
M_inv = cv2.invertAffineTransform(M)

dst_grid = np.stack([xs.ravel(), ys.ravel(), np.ones(xs.size)])
src_coords = M_inv @ dst_grid
src_x = src_coords[0].reshape(h, w)
src_y = src_coords[1].reshape(h, w)

print('example fractional source coordinate for destination pixel (10, 15):')
print(f'  ({src_x[15, 10]:.3f}, {src_y[15, 10]:.3f})')

Ảnh động này minh họa ánh xạ ngược: quét ảnh *đích* theo từng hàng (với cách lấy mẫu láng giềng gần nhất, sẽ giới thiệu ngay sau đây). Khung bên trái cho thấy các điểm ảnh của mỗi hàng đích lấy mẫu từ đâu trong ảnh nguồn -- hãy để ý rằng đó là một đường quét chéo qua các tọa độ lẻ, không phải một lưới ngay ngắn. Ảnh đích được lấp đầy hoàn toàn, không có lỗ hổng, trái ngược hẳn với ảnh động ánh xạ xuôi ở trên.

<img src="../animations/inverse_mapping_demo.gif" alt="Ảnh động biến dạng bằng ánh xạ ngược, quét ảnh đích theo từng hàng và lấy mẫu mỗi điểm ảnh từ tọa độ nguồn lẻ của nó, lấp đầy ảnh đích hoàn toàn không có lỗ hổng" style="display: block; margin: 0 auto; max-width: 680px;">

### Nội suy láng giềng gần nhất (Nearest-neighbor interpolation)
Lựa chọn đơn giản nhất: chỉ cần làm tròn tới điểm ảnh nguồn nguyên gần nhất. Cách này không có lỗ hổng, nhưng bị vỡ khối &mdash; nhiều điểm ảnh đích làm tròn về *cùng một* điểm ảnh nguồn, và ảnh trông bị rỗ điểm (pixelated) ở những chỗ phép biến đổi phóng to ảnh.

In [ ]:
def nearest_sample(image, xf, yf):
    h, w = image.shape[:2]
    xi = np.round(xf).astype(int)
    yi = np.round(yf).astype(int)
    valid = (xi >= 0) & (xi < w) & (yi >= 0) & (yi < h)
    out = np.zeros(xf.shape + (image.shape[2],), dtype=np.uint8)
    out[valid] = image[yi[valid], xi[valid]]
    return out

nearest = nearest_sample(img, src_x, src_y)

plt.imshow(nearest)
plt.title('Inverse mapping + nearest neighbor: no holes, but blocky')
plt.axis('off')
plt.show()

### Nội suy song tuyến tính (Bilinear interpolation)
Thay vì bám vào một điểm ảnh gần nhất duy nhất, **nội suy song tuyến tính (bilinear interpolation)** trộn 4 điểm ảnh lân cận, đánh trọng số theo độ gần tới mỗi điểm. Đặt $(x_0,y_0)$ là điểm ảnh nguyên ngay bên dưới-trái của $(x,y)$, và đặt $d_x = x-x_0$, $d_y=y-y_0$, trong đó $0 \le d_x, dy < 1$ là các tọa độ lẻ. Khi đó giá trị nội suy là:

$$I(x,y) \approx (1-d_x)(1-d_y)\,I(x_0,y_0) + d_x(1-d_y)\,I(x_1,y_0) + (1-d_x)\,d_y\,I(x_0,y_1) + d_x\,d_y\,I(x_1,y_1)$$

trong đó $x_1=x_0+1$, $y_1=y_0+1$. Đây chỉ là hai phép nội suy tuyến tính 1D (theo $x$, rồi theo $y$) ghép lại &mdash; vì thế gọi là *song* tuyến tính.

In [ ]:
def bilinear_sample(image, xf, yf):
    h, w = image.shape[:2]
    x0 = np.floor(xf).astype(int)
    y0 = np.floor(yf).astype(int)
    x1, y1 = x0 + 1, y0 + 1
    dx = (xf - x0)[..., None]
    dy = (yf - y0)[..., None]

    valid = (x0 >= 0) & (x1 < w) & (y0 >= 0) & (y1 < h)
    x0c, x1c = np.clip(x0, 0, w - 1), np.clip(x1, 0, w - 1)
    y0c, y1c = np.clip(y0, 0, h - 1), np.clip(y1, 0, h - 1)

    Ia = image[y0c, x0c].astype(np.float64)
    Ib = image[y0c, x1c].astype(np.float64)
    Ic = image[y1c, x0c].astype(np.float64)
    Id = image[y1c, x1c].astype(np.float64)

    out = (1 - dx) * (1 - dy) * Ia + dx * (1 - dy) * Ib + (1 - dx) * dy * Ic + dx * dy * Id
    out[~valid] = 0
    return np.clip(out, 0, 255).astype(np.uint8)

bilinear = bilinear_sample(img, src_x, src_y)

fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
axes[0].imshow(nearest)
axes[0].set_title('Nearest neighbor')
axes[0].axis('off')
axes[1].imshow(bilinear)
axes[1].set_title('Bilinear')
axes[1].axis('off')
plt.tight_layout()
plt.show()

Ảnh động này minh họa cách đánh trọng số của nội suy song tuyến tính, dùng một vùng lân cận 2x2 nhỏ nằm vắt qua cạnh của hình tam giác (để 4 góc không cùng một màu). Trọng số của mỗi góc -- và màu trộn thu được -- cập nhật trực tiếp khi điểm lấy mẫu lẻ (dx, dy) di chuyển xung quanh.

<img src="../animations/bilinear_demo.gif" alt="Ảnh động nội suy song tuyến tính: một vị trí lấy mẫu quét quanh vùng lân cận 2x2 điểm ảnh, với trọng số của mỗi góc và màu trộn thu được cập nhật trực tiếp" style="display: block; margin: 0 auto; max-width: 680px;">

Kết quả song tuyến tính có các cạnh mượt quanh hình tam giác và hình chữ nhật thay vì các bậc răng cưa &mdash; sự đánh đổi kinh điển là ảnh mềm hơn, hơi mờ hơn để đổi lấy việc loại bỏ các nhiễu răng cưa (aliasing artifacts).

### Đối chiếu với OpenCV (Sanity check against OpenCV)
`cv2.warpAffine` thực hiện đúng việc này &mdash; ánh xạ ngược cộng với nội suy &mdash; ở bên trong. Ta so sánh phiên bản tự viết với `cv2.warpAffine(..., flags=cv2.INTER_LINEAR)` áp dụng với ma trận *xuôi* `M` (theo mặc định OpenCV tự nghịch đảo nó bên trong).

In [ ]:
cv_result = cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_LINEAR)

diff = np.abs(bilinear.astype(int) - cv_result.astype(int))
print(f'max pixel difference  = {diff.max()}   (out of 255)')
print(f'mean pixel difference = {diff.mean():.3f}')

fig, axes = plt.subplots(1, 3, figsize=(9, 3.5))
for ax, im, title in zip(axes, [bilinear, cv_result, diff.astype(np.uint8) * 20],
                          ['Our bilinear', 'cv2.warpAffine', 'Difference (x20)']):
    ax.imshow(im)
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

Hai kết quả khớp gần như hoàn toàn &mdash; khác biệt rất nhỏ còn lại (tối đa 1 mức xám) đến từ việc OpenCV dùng phép làm tròn dấu phẩy tĩnh (fixed-point) bên trong để tăng tốc, thay vì số học dấu phẩy động đầy đủ.

### Bài tập (Exercises)
1. Tại sao vấn đề lỗ hổng của ánh xạ xuôi không xảy ra khi phép biến đổi chỉ *thu nhỏ* ảnh? Hãy thử hệ số co giãn `0.5` thay vì `1.6` trong ví dụ ánh xạ xuôi và xem bao nhiêu phần trăm điểm ảnh đích được lấp đầy.
2. Mở rộng `bilinear_sample` thành nội suy song bậc ba (bicubic) về mặt ý tưởng: thay vì vùng lân cận 2x2, bạn cần vùng lân cận kích thước bao nhiêu, và bạn muốn các trọng số thỏa mãn tính chất gì tại các điểm lấy mẫu?
3. Đo thời gian `bilinear_sample` so với `cv2.warpAffine` trên một ảnh lớn hơn (ví dụ 800x800) bằng `%timeit`. Phiên bản OpenCV nhanh hơn bao nhiêu, và tại sao?